# 02 | Exploration

This notebook covers the first descriptive questions:

- How do popularity and prestige relate?
- How unequal is attention on the profile side?
- Which profiles and raters stand out most strongly?


## Setup

If the required artifacts are missing, this notebook rebuilds them with the project analysis script.
Otherwise it reuses the cached outputs.


In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


def locate_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "src" / "rec_dating_project").exists():
            return candidate
    raise RuntimeError("Could not locate the project root from the current working directory.")


project_root = locate_project_root()
sys.path.insert(0, str(project_root / "src"))

from rec_dating_project import PopularityPrestigeAnalyzer, ProjectPaths


paths = ProjectPaths.default()
paths.ensure_output_dirs()
OUTPUT_DATA = paths.output_data_dir
OUTPUT_FIGURES = paths.output_figures_dir
FORCE_REBUILD = False


def run_script(script_name: str, *args: str) -> None:
    cmd = [sys.executable, str(paths.scripts_dir / script_name), *args]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=paths.project_root)


def ensure_dataset_summary(force: bool = False) -> None:
    summary_path = OUTPUT_DATA / "dataset_summary.json"
    if force or not summary_path.exists():
        run_script("01_dataset_overview.py")


def ensure_first_study_outputs(force: bool = False) -> None:
    ensure_dataset_summary(force=force)
    required_data = [
        OUTPUT_DATA / "dataset_summary.json",
        OUTPUT_DATA / "rating_distribution_full.csv",
        OUTPUT_DATA / "profile_metrics_full.csv",
        OUTPUT_DATA / "profile_metrics_positive_8_full.csv",
        OUTPUT_DATA / "rater_metrics_full.csv",
        OUTPUT_DATA / "top_profiles_popularity_full.csv",
        OUTPUT_DATA / "top_profiles_prestige_full.csv",
        OUTPUT_DATA / "top_profiles_prestige_gap_positive_full.csv",
        OUTPUT_DATA / "top_profiles_prestige_gap_negative_full.csv",
        OUTPUT_DATA / "top_raters_activity_full.csv",
        OUTPUT_DATA / "top_raters_hub_full.csv",
    ]
    required_figures = [
        OUTPUT_FIGURES / "rating_distribution_full.png",
        OUTPUT_FIGURES / "strength_ccdf_full.png",
        OUTPUT_FIGURES / "popularity_vs_prestige_full.png",
        OUTPUT_FIGURES / "popularity_vs_prestige_2_full.png",
        OUTPUT_FIGURES / "profile_lorenz_full.png",
    ]
    missing = [path for path in required_data + required_figures if not path.exists()]
    if force or missing:
        run_script("02_full_project_analysis.py")


ensure_first_study_outputs(FORCE_REBUILD)

summary = json.loads((OUTPUT_DATA / "dataset_summary.json").read_text(encoding="utf-8"))
rating_distribution = pd.read_csv(OUTPUT_DATA / "rating_distribution_full.csv")
profile_metrics = pd.read_csv(OUTPUT_DATA / "profile_metrics_full.csv")
positive_profile_metrics = pd.read_csv(OUTPUT_DATA / "profile_metrics_positive_8_full.csv")
rater_metrics = pd.read_csv(OUTPUT_DATA / "rater_metrics_full.csv")
top_popularity = pd.read_csv(OUTPUT_DATA / "top_profiles_popularity_full.csv")
top_prestige = pd.read_csv(OUTPUT_DATA / "top_profiles_prestige_full.csv")
top_gap_positive = pd.read_csv(OUTPUT_DATA / "top_profiles_prestige_gap_positive_full.csv")
top_gap_negative = pd.read_csv(OUTPUT_DATA / "top_profiles_prestige_gap_negative_full.csv")
top_raters_activity = pd.read_csv(OUTPUT_DATA / "top_raters_activity_full.csv")
top_raters_hub = pd.read_csv(OUTPUT_DATA / "top_raters_hub_full.csv")


def top_overlap(frame: pd.DataFrame, col_a: str, col_b: str, k: int = 100) -> dict[str, float]:
    a = set(frame.sort_values(col_a, ascending=False).head(k)["profile_id"].tolist())
    b = set(frame.sort_values(col_b, ascending=False).head(k)["profile_id"].tolist())
    inter = len(a & b)
    union = len(a | b)
    return {"k": k, "intersection": inter, "jaccard": (inter / union) if union else 0.0}


full_corr = {
    "pearson": float(profile_metrics["in_strength"].corr(profile_metrics["authority_score"], method="pearson")),
    "spearman": float(profile_metrics["in_strength"].corr(profile_metrics["authority_score"], method="spearman")),
}
pos_corr = {
    "pearson": float(positive_profile_metrics["in_strength"].corr(positive_profile_metrics["authority_score"], method="pearson")),
    "spearman": float(positive_profile_metrics["in_strength"].corr(positive_profile_metrics["authority_score"], method="spearman")),
}
overlap = top_overlap(profile_metrics, "in_strength", "authority_score", k=100)
profile_ineq = {
    "gini_in_strength": PopularityPrestigeAnalyzer.gini(profile_metrics["in_strength"]),
    "gini_authority": PopularityPrestigeAnalyzer.gini(profile_metrics["authority_score"]),
    "top_1pct_in_strength_share": PopularityPrestigeAnalyzer.top_share(profile_metrics["in_strength"], fraction=0.01),
    "top_1pct_authority_share": PopularityPrestigeAnalyzer.top_share(profile_metrics["authority_score"], fraction=0.01),
}


## Core Exploration Summary

The table below collects the main descriptive numbers from the first study.


In [ ]:
exploration_summary = pd.DataFrame(
    [
        {"metric": "Full-layer Pearson correlation", "value": full_corr["pearson"]},
        {"metric": "Full-layer Spearman correlation", "value": full_corr["spearman"]},
        {"metric": "Positive-layer Pearson correlation (rating >= 8)", "value": pos_corr["pearson"]},
        {"metric": "Positive-layer Spearman correlation (rating >= 8)", "value": pos_corr["spearman"]},
        {"metric": "Top-100 popularity/prestige overlap", "value": overlap["intersection"]},
        {"metric": "Top-100 popularity/prestige Jaccard", "value": overlap["jaccard"]},
        {"metric": "Gini of profile in-strength", "value": profile_ineq["gini_in_strength"]},
        {"metric": "Gini of profile authority", "value": profile_ineq["gini_authority"]},
        {"metric": "Top 1% share of in-strength", "value": profile_ineq["top_1pct_in_strength_share"]},
        {"metric": "Top 1% share of authority", "value": profile_ineq["top_1pct_authority_share"]},
    ]
)

display(exploration_summary)


## Visual Exploration

We now inspect the key figures one by one.

The first two plots show how the rating scale is used and how heavy-tailed the interaction distribution is.
The next three focus on popularity, prestige, and inequality on the profile side.


In [ ]:
figure_notes = [
    (
        "rating_distribution_full.png",
        "The rating scale is used unevenly, with substantial mass near the upper end.",
    ),
    (
        "strength_ccdf_full.png",
        "Interaction is heavy-tailed on both sides of the bipartite graph, especially on the profile side.",
    ),
    (
        "popularity_vs_prestige_full.png",
        "The raw scatter shows that more popular profiles are usually also more prestigious.",
    ),
    (
        "popularity_vs_prestige_2_full.png",
        "The percentile-binned version gives a cleaner summary of the same relationship.",
    ),
    (
        "profile_lorenz_full.png",
        "Both attention and prestige are highly unequal, and raw attention is slightly more concentrated than authority in the summary statistics.",
    ),
]

for filename, note in figure_notes:
    display(Markdown(f"### {filename}\n\n{note}"))
    display(Image(filename=str(OUTPUT_FIGURES / filename)))


## Concrete Rankings

Exploratory analysis should not stop at correlations.
The tables below show which concrete profiles and raters occupy the most visible positions in the network.


In [ ]:
display(Markdown("### Most popular profiles"))
display(top_popularity.head(10))

display(Markdown("### Most prestigious profiles"))
display(top_prestige.head(10))

display(Markdown("### Profiles whose prestige is higher than their popularity rank would suggest"))
display(top_gap_positive.head(10))

display(Markdown("### Profiles whose popularity is higher than their prestige rank would suggest"))
display(top_gap_negative.head(10))

display(Markdown("### Most active raters"))
display(top_raters_activity.head(10))

display(Markdown("### Strongest hub raters"))
display(top_raters_hub.head(10))


In [ ]:
display(
    Markdown(
        f"""
        ## Main Takeaways

        - Popularity and prestige are strongly aligned in the full network (**Pearson = {full_corr['pearson']:.3f}**, **Spearman = {full_corr['spearman']:.3f}**).
        - The alignment remains strong when we restrict the graph to positive ratings of at least 8.
        - The rankings are not identical: the top-100 sets overlap on **{overlap['intersection']}** profiles, not all 100.
        - Both attention and prestige are highly unequal; the inequality summary suggests that **raw attention is slightly more concentrated than authority** in the full-layer outputs.
        """
    )
)


## Next

The next notebook moves from broad description to the concentration and feature-alignment analyses:

- concentration of high and low ratings
- concentration of overall interaction
- feature alignment with the popularity/prestige results
